# Chronos: Временной ряд как текст

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/31_chronos.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q chronos-forecasting torch pandas numpy matplotlib

## Подготовка данных

In [ ]:
import torch
import numpy as np
import pandas as pd

# Создаём синтетические данные
np.random.seed(42)
t = np.arange(200)
y = 100 + np.cumsum(np.random.randn(200)) + 20 * np.sin(t / 7 * 2 * np.pi)

# Подготовка для Chronos
context = torch.tensor(y).unsqueeze(0).float()
print(f"Context shape: {context.shape}")

## Chronos Bolt (рекомендуется)

In [ ]:
from chronos import BaseChronosPipeline

# Загрузка Bolt модели
pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-small",
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.float32,
)

print(f"Модель загружена на устройство: {pipeline.device}")

In [ ]:
# Прогнозирование с Bolt
prediction_length = 24

quantiles, mean = pipeline.predict_quantiles(
    context.to(pipeline.device),
    prediction_length=prediction_length,
    quantile_levels=[0.1, 0.25, 0.5, 0.75, 0.9],
)

print(f"Quantiles shape: {quantiles.shape}")
print(f"Mean shape: {mean.shape}")

# Доступ к конкретным квантилям
p10 = quantiles[0, 0, :].cpu().numpy()  # 10-й перцентиль
p50 = quantiles[0, 2, :].cpu().numpy()  # медиана
p90 = quantiles[0, 4, :].cpu().numpy()  # 90-й перцентиль

## Визуализация прогноза

In [ ]:
import matplotlib.pyplot as plt

def plot_chronos_forecast(context, mean, quantiles, quantile_levels, title=''):
    """Визуализация прогноза Chronos с интервалами."""
    
    fig, ax = plt.subplots(figsize=(12, 5))
    
    context_np = context[0].cpu().numpy()
    mean_np = mean[0].cpu().numpy()
    quantiles_np = quantiles[0].cpu().numpy()
    
    # История
    history_idx = range(len(context_np))
    ax.plot(history_idx, context_np, 'b-', linewidth=2, label='История')
    
    # Прогноз
    forecast_start = len(context_np)
    horizon = len(mean_np)
    forecast_idx = range(forecast_start, forecast_start + horizon)
    
    # Находим индексы квантилей для интервалов
    q_dict = {q: i for i, q in enumerate(quantile_levels)}
    
    # 80% интервал (p10 и p90)
    if 0.1 in q_dict and 0.9 in q_dict:
        ax.fill_between(
            forecast_idx,
            quantiles_np[q_dict[0.1]],
            quantiles_np[q_dict[0.9]],
            alpha=0.2, color='red', label='80% интервал'
        )
    
    # 50% интервал (p25 и p75)
    if 0.25 in q_dict and 0.75 in q_dict:
        ax.fill_between(
            forecast_idx,
            quantiles_np[q_dict[0.25]],
            quantiles_np[q_dict[0.75]],
            alpha=0.3, color='red', label='50% интервал'
        )
    
    # Медиана или mean
    ax.plot(forecast_idx, mean_np, 'r-', linewidth=2, label='Прогноз')
    
    ax.axvline(x=forecast_start, color='gray', linestyle='--', alpha=0.5)
    ax.legend()
    ax.set_title(title or 'Chronos Bolt: прогноз')
    ax.set_xlabel('Время')
    ax.set_ylabel('Значение')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_chronos_forecast(
    context, mean, quantiles, 
    quantile_levels=[0.1, 0.25, 0.5, 0.75, 0.9],
    title='Chronos Bolt: прогноз временного ряда'
)

## Работа с несколькими рядами

In [ ]:
# Создаём несколько рядов
n_series = 5
series_list = []

for i in range(n_series):
    np.random.seed(i)
    t = np.arange(150)
    y = 100 + np.cumsum(np.random.randn(150)) + 20 * np.sin(t / 7 * 2 * np.pi)
    series_list.append(y)

# Батч для Chronos
batch_context = torch.tensor(np.array(series_list)).float()
print(f"Batch context shape: {batch_context.shape}")

# Прогноз для всех рядов
batch_quantiles, batch_mean = pipeline.predict_quantiles(
    batch_context.to(pipeline.device),
    prediction_length=24,
    quantile_levels=[0.1, 0.5, 0.9],
)

print(f"Batch quantiles shape: {batch_quantiles.shape}")
print(f"Batch mean shape: {batch_mean.shape}")

## Визуализация нескольких прогнозов

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i in range(min(n_series, 6)):
    ax = axes[i]
    
    history = batch_context[i].numpy()
    forecast_mean = batch_mean[i].cpu().numpy()
    forecast_p10 = batch_quantiles[i, 0, :].cpu().numpy()
    forecast_p90 = batch_quantiles[i, 2, :].cpu().numpy()
    
    # История
    ax.plot(range(len(history)), history, 'b-', label='История')
    
    # Прогноз
    forecast_start = len(history)
    forecast_idx = range(forecast_start, forecast_start + len(forecast_mean))
    
    ax.fill_between(forecast_idx, forecast_p10, forecast_p90, 
                    alpha=0.3, color='red', label='80% интервал')
    ax.plot(forecast_idx, forecast_mean, 'r-', label='Прогноз')
    
    ax.axvline(x=forecast_start, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'Series {i}')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

# Скрываем пустой subplot если n_series < 6
for i in range(n_series, 6):
    axes[i].set_visible(False)

plt.suptitle('Chronos Bolt: batch прогнозирование', fontsize=14)
plt.tight_layout()
plt.show()

## Работа с pandas DataFrame

In [ ]:
def chronos_bolt_forecast_df(df, pipeline, prediction_length, 
                              quantile_levels=[0.1, 0.5, 0.9]):
    """
    Прогнозирование Chronos Bolt для DataFrame.
    """
    results = []
    
    for uid in df['unique_id'].unique():
        series = df[df['unique_id'] == uid].sort_values('ds')['y'].values
        context = torch.tensor(series).unsqueeze(0).float()
        
        # Прогноз
        quantiles, mean = pipeline.predict_quantiles(
            context.to(pipeline.device),
            prediction_length=prediction_length,
            quantile_levels=quantile_levels
        )
        
        quantiles_np = quantiles[0].cpu().numpy()
        mean_np = mean[0].cpu().numpy()
        
        for h in range(prediction_length):
            result = {
                'unique_id': uid,
                'horizon': h + 1,
                'forecast': mean_np[h],
            }
            for i, q in enumerate(quantile_levels):
                result[f'p{int(q*100)}'] = quantiles_np[i, h]
            results.append(result)
    
    return pd.DataFrame(results)

# Создаём тестовый DataFrame
dates = pd.date_range('2023-01-01', periods=100, freq='D')
test_df = pd.DataFrame({
    'unique_id': np.repeat(['A', 'B', 'C'], 100),
    'ds': np.tile(dates, 3),
    'y': np.random.randn(300).cumsum() + 100
})

# Прогноз
forecast_df = chronos_bolt_forecast_df(
    test_df, pipeline, 
    prediction_length=16,
    quantile_levels=[0.1, 0.25, 0.5, 0.75, 0.9]
)

print(forecast_df.head(20))